# ResNet18로 개 vs 고양이 분류하기 — 전이학습 vs 밑바닥 학습

이 노트북 하나로 **Google Colab에서도, 내 노트북에서도** 그대로 돌아갑니다.
필요한 라이브러리 설치와 데이터 다운로드까지 전부 안에 들어 있습니다.

## 무엇을 하나요?

ImageNet 사진 128만 장으로 미리 훈련된 **ResNet18**을 데려와서, 마지막 판정 층만
"개냐 고양이냐" 2지선다로 갈아끼우고 조금만 더 가르칩니다. 그리고 같은 조건에서
**아무것도 모르는 상태(랜덤 가중치)** 로 처음부터 가르친 모델과 성적을 비교합니다.

> **비유**: 사진을 수백만 장 봐 온 사진 감별사가 있습니다. 이 사람은 이미 털·귀·눈·질감을
> 구분하는 눈을 갖고 있어요. 여기에 "이제부터 개랑 고양이만 골라주세요"라고 며칠만 알려주면
> 금방 잘합니다(**전이학습**). 반대로 사진을 한 번도 본 적 없는 사람에게 같은 시간을 주면
> 아직 "털이 뭔지"부터 배우느라 개/고양이 구분은 시작도 못 합니다(**밑바닥 학습**).

## 실험 3가지

| 조건 | 시작 가중치 | 학습하는 부분 |
|---|---|---|
| `pretrained` | ImageNet 사전학습 | 마지막 층 교체 후 **전체** 미세조정 |
| `scratch` | 랜덤 | 전체 |
| `linear-probe` (보너스) | ImageNet 사전학습 | **마지막 층만** (나머지는 동결) |

## 목차

0. 환경 준비
1. 데이터셋 소개
2. 전처리와 데이터 증강
3. 데이터 내려받기 & 라벨 살펴보기
4. 일부만 골라 쓰기 (층화 추출)
5. 사진 직접 보기
6. 모델 만들기 — 마지막 층 갈아끼우기
7. 학습·평가 루프
8. 실험 1 · 2 · 3
9. 결과 비교
10. 틀린 사진 들여다보기
11. 정리

---
## 0. 환경 준비

### Colab에서 GPU 켜기

상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 를 먼저 선택하세요.
GPU 없이 CPU로 돌리면 한 실험에 10분 이상 걸릴 수 있습니다.

아래 셀은 `torch`가 없으면 설치합니다. Colab에는 이미 깔려 있어서 보통 그냥 넘어갑니다.

In [ ]:
# torch / torchvision 이 없으면 설치 (Colab에는 대개 이미 있음)
try:
    import torch, torchvision
    print(f"이미 설치됨 → torch {torch.__version__} / torchvision {torchvision.__version__}")
except ImportError:
    print("torch 가 없어 설치합니다 (몇 분 걸립니다)")
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision"])
    import torch, torchvision
    print(f"설치 완료 → torch {torch.__version__} / torchvision {torchvision.__version__}")

# 그래프용
try:
    import matplotlib
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])

### 라이브러리 불러오기 + 재현성 확보

같은 코드를 두 번 돌렸을 때 결과가 달라지면 "사전학습이 좋아서 이긴 건지, 운이 좋아서 이긴 건지"
알 수 없습니다. 그래서 **난수 시드를 고정**합니다. 세 실험 모두 같은 시드에서 출발하므로
초기화·데이터 순서·증강이 동일해지고, 차이는 오직 *가중치를 어디서 시작했는가* 에서만 나옵니다.

In [1]:
import json, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from torchvision.models import resnet18, ResNet18_Weights

import matplotlib
import matplotlib.pyplot as plt


def set_seed(seed=42):
    # 파이썬 / 넘파이 / 토치의 난수를 한꺼번에 고정
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device():
    # CUDA(Colab GPU) > MPS(Apple Silicon) > CPU 순으로 고른다
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


SEED = 42
set_seed(SEED)
DEVICE = pick_device()

print("사용할 장치:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
elif DEVICE.type == "cpu":
    print("경고: CPU로 돌아갑니다. Colab이라면 [런타임 > 런타임 유형 변경]에서 GPU를 켜세요.")

사용할 장치: mps


### 실험 설정값

여기 숫자만 바꾸면 전체 실험 규모가 달라집니다. 처음엔 기본값으로 돌려보고,
시간이 남으면 `TRAIN_SIZE = 0`(전체 사용)이나 `EPOCHS`를 늘려보세요.

In [ ]:
# ── 실험 설정 ────────────────────────────────────────────────
DATA_ROOT   = "./data"   # 데이터 저장 위치 (Colab이면 세션 종료 시 사라짐)
OUT_DIR     = "./outputs"

TRAIN_SIZE  = 2000       # 학습에 쓸 이미지 수 (0 이면 전체 3,680장)
TEST_SIZE   = 1000       # 시험에 쓸 이미지 수 (0 이면 전체 3,669장)
EPOCHS      = 5          # 세 실험 모두 동일하게 적용
BATCH_SIZE  = 32
IMG_SIZE    = 224        # ResNet이 ImageNet에서 학습된 해상도
NUM_WORKERS = 2          # Colab은 2가 무난

LR_PRETRAINED = 3e-4     # 이미 잘하는 모델은 살살 (크게 흔들면 배운 걸 잊는다)
LR_SCRATCH    = 1e-3     # 맨땅에서 시작하는 쪽은 더 크게 (불리하지 않도록 배려)
LR_LINEAR     = 1e-3     # 마지막 층만 학습하니 크게 줘도 안전

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print("설정 완료")

---
## 1. 데이터셋 — Oxford-IIIT Pet

**왜 이 데이터셋인가?**

- torchvision에 내장돼 있어 `download=True` 한 줄이면 끝납니다.
- 옥스퍼드대가 공개 배포하므로 **로그인·API 토큰·약관 동의가 전혀 필요 없습니다.**
  (Kaggle의 Dogs vs. Cats는 계정과 API 키가 필요해서 피했습니다.)
- 개 25품종 + 고양이 12품종 = 37품종, 총 7,349장. `trainval` 3,680장 / `test` 3,669장으로
  **공식 분할이 이미 되어 있어** 우리가 임의로 나눌 필요가 없습니다.

**핵심 옵션은 `target_types="binary-category"`** 입니다.
품종 37종짜리 라벨 대신 **0 = Cat, 1 = Dog** 이진 라벨을 바로 내려줍니다. 우리가 품종명을
일일이 개/고양이로 매핑할 필요가 없다는 뜻이에요.

> 최초 실행 시 이미지 약 **792MB**를 내려받습니다. Colab 기준 몇 분 걸리고,
> 한 번 받으면 같은 세션 안에서는 다시 받지 않습니다.

---
## 2. 전처리와 데이터 증강

사진 크기가 제각각이라 그대로는 신경망에 넣을 수 없습니다. 세 가지를 합니다.

**① 크기 맞추기** — 모두 224×224로. ResNet18이 ImageNet에서 이 크기로 학습됐기 때문입니다.

**② 정규화(Normalize)** — ImageNet 전체의 채널별 평균/표준편차로 픽셀값을 보정합니다.
사전학습 가중치는 "이렇게 보정된 입력"을 전제로 만들어졌으므로, 같은 기준을 써야 제 실력이 나옵니다.
공정한 비교를 위해 `scratch` 실험에도 똑같은 전처리를 적용합니다.

**③ 증강(Augmentation) — 학습용에만** — 매번 조금씩 다르게 자르고(`RandomResizedCrop`)
좌우를 뒤집습니다(`RandomHorizontalFlip`). 같은 사진을 매번 다른 각도로 보여주는 셈이라,
모델이 사진을 통째로 외워버리는 **과적합**을 늦춥니다.

시험용에는 증강을 쓰지 않습니다. 채점 기준은 매번 똑같아야 하니까요.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)   # ImageNet 데이터의 R,G,B 채널 평균
IMAGENET_STD  = (0.229, 0.224, 0.225)   # 채널 표준편차

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),  # 70~100% 영역을 랜덤하게 잘라 확대
    transforms.RandomHorizontalFlip(),                          # 50% 확률로 좌우 반전
    transforms.ToTensor(),                                      # PIL 이미지 → [0,1] 텐서
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),  # 256으로 줄인 뒤
    transforms.CenterCrop(IMG_SIZE),          # 가운데 224만 사용 (항상 같은 방식 = 재현 가능)
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("학습용 변환:", len(train_tf.transforms), "단계  |  시험용 변환:", len(eval_tf.transforms), "단계")

---
## 3. 데이터 내려받기 & 라벨 살펴보기

`download=True`는 **이미 받아둔 파일이 있으면 건너뜁니다.** 그래서 이 셀은 몇 번 실행해도 안전합니다.

내려받은 뒤에는 반드시 **클래스 분포**를 확인하세요. 개가 고양이보다 두 배 많기 때문에,
"무조건 개라고 찍기"만 해도 정확도가 약 68% 나옵니다. 이 값이 우리의 **기준선(baseline)** 이고,
모델 성적은 항상 이 숫자와 비교해서 읽어야 합니다. 68%는 잘한 게 아니라 아무것도 안 한 것이니까요.

In [ ]:
Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)

# 이미 받아둔 게 있는지 먼저 확인 (Colab 재실행 시 시간 절약)
already = (Path(DATA_ROOT) / "oxford-iiit-pet" / "images").exists()
print("기존 데이터 발견 → 다운로드 건너뜀" if already else "데이터가 없어 내려받습니다 (약 792MB, 몇 분 소요)")

common = dict(root=DATA_ROOT, target_types="binary-category", download=True)
train_full = OxfordIIITPet(split="trainval", transform=train_tf, **common)
test_full  = OxfordIIITPet(split="test",     transform=eval_tf,  **common)

CLASS_NAMES = ["Cat", "Dog"]   # binary-category: 0=Cat, 1=Dog
print(f"\ntrainval {len(train_full)}장 / test {len(test_full)}장")

### 라벨을 빠르게 꺼내는 요령

`dataset[i]`로 하나씩 꺼내면 **이미지 파일을 실제로 디코딩**하기 때문에 7천 장이면 한참 걸립니다.
라벨만 필요할 땐 데이터셋이 내부에 들고 있는 `_bin_labels` 리스트를 바로 읽습니다.
(torchvision 버전에 따라 이름이 다를 수 있어 아래 함수는 대비책까지 넣어 뒀습니다.)

In [ ]:
def binary_labels(ds):
    # 이미지 로딩 없이 이진 라벨(0=Cat, 1=Dog) 배열만 꺼낸다
    for attr in ("_bin_labels", "_binary_labels"):
        if hasattr(ds, attr):
            return np.asarray(getattr(ds, attr))
    # 폴백: Oxford-IIIT Pet 규칙상 고양이 품종 파일명만 대문자로 시작한다
    return np.asarray([0 if Path(p).name[0].isupper() else 1 for p in ds._images])


tr_all, te_all = binary_labels(train_full), binary_labels(test_full)
for name, lab in [("trainval", tr_all), ("test", te_all)]:
    n_cat, n_dog = int((lab == 0).sum()), int((lab == 1).sum())
    print(f"{name:9s}  Cat {n_cat:5d}  Dog {n_dog:5d}   (개 비율 {n_dog / len(lab) * 100:.1f}%)")

print(f"\n★ 기준선: test에서 전부 Dog로 찍으면 {(te_all == 1).mean() * 100:.2f}%")

---
## 4. 일부만 골라 쓰기 — 층화 추출(stratified sampling)

시간을 아끼려고 전체가 아닌 일부만 씁니다. 그런데 **아무렇게나 뽑으면 안 됩니다.**
운 나쁘게 개만 잔뜩 뽑히면 모델이 "다 개야"라고 답해도 점수가 높게 나와서, 실험 자체가 무의미해집니다.

그래서 원본의 클래스 비율(개 약 2 : 고양이 1)을 **그대로 유지**하면서 뽑습니다. 이것이 층화 추출입니다.

In [ ]:
def stratified_subset(ds, n, seed):
    # 클래스 비율을 유지하면서 n장만 뽑는다 (n <= 0 이면 전체)
    labels = binary_labels(ds)
    if n <= 0 or n >= len(labels):
        return Subset(ds, list(range(len(labels))))

    rng = np.random.default_rng(seed)
    picked = []
    for cls in (0, 1):
        idx = np.flatnonzero(labels == cls)
        take = round(n * len(idx) / len(labels))          # 원본 비율만큼 배분
        picked += rng.choice(idx, size=min(take, len(idx)), replace=False).tolist()
    rng.shuffle(picked)
    return Subset(ds, picked)


train_ds = stratified_subset(train_full, TRAIN_SIZE, SEED)
test_ds  = stratified_subset(test_full,  TEST_SIZE,  SEED + 1)   # 학습과 다른 시드

tr_lab = tr_all[train_ds.indices]
te_lab = te_all[test_ds.indices]
BASELINE = (te_lab == 1).mean()

print(f"학습 {len(train_ds)}장 (Cat {int((tr_lab==0).sum())} / Dog {int((tr_lab==1).sum())})")
print(f"시험 {len(test_ds)}장 (Cat {int((te_lab==0).sum())} / Dog {int((te_lab==1).sum())})")
print(f"기준선(항상 Dog) = {BASELINE*100:.2f}%  ← 이 숫자를 못 넘으면 학습 실패")

### DataLoader — 배치로 묶어 공급하기

`DataLoader`는 데이터셋에서 `BATCH_SIZE`장씩 묶어 꺼내주는 컨베이어 벨트입니다.

- `shuffle=True` (학습): 매 에폭 순서를 섞습니다. 같은 순서로만 보면 순서까지 외워버립니다.
- `shuffle=False` (시험): 채점은 항상 같은 순서로.
- `num_workers`: 사진을 읽고 변환하는 일을 맡을 별도 프로세스 수.

In [ ]:
loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                 pin_memory=(DEVICE.type == "cuda"),
                 persistent_workers=NUM_WORKERS > 0)

train_loader = DataLoader(train_ds, shuffle=True,  **loader_kw)
test_loader  = DataLoader(test_ds,  shuffle=False, **loader_kw)

xb, yb = next(iter(train_loader))
print("배치 한 개 모양:", tuple(xb.shape), "  라벨:", tuple(yb.shape))
print("→ (배치 32장, RGB 3채널, 224×224 픽셀)")
print(f"1 에폭 = {len(train_loader)} 배치")

---
## 5. 사진 직접 보기

데이터를 눈으로 확인하지 않고 학습부터 돌리는 건 위험합니다. 라벨이 뒤집혀 있거나
전처리가 망가져 있어도 손실값만 보면 알아채기 어렵거든요. 정규화를 되돌려서 원래 색으로 봅시다.

In [ ]:
def denormalize(t):
    # Normalize를 되돌려 사람이 볼 수 있는 이미지로
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, img, lab in zip(axes.flat, xb, yb):
    ax.imshow(denormalize(img))
    ax.set_title(CLASS_NAMES[lab.item()], fontsize=11)
    ax.axis("off")
fig.suptitle("Training batch (augmented)", fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. 모델 만들기 — 마지막 층만 갈아끼우기

ResNet18은 크게 두 부분입니다.

```
입력 사진 → [ 백본(backbone): conv 층들 ] → 512차원 특징 벡터 → [ fc: 최종 판정 층 ] → 클래스 점수
              ↑ 여기서 털·눈·귀·질감을 읽어낸다          ↑ 여기서 "무엇인지" 결론
```

ImageNet 사전학습 모델의 `fc`는 **1000개 클래스**(비행기, 컵, 골든리트리버…)를 위한 층입니다.
우리는 2개(Cat/Dog)만 필요하니, **`fc`만 새 것으로 교체**합니다. 백본은 그대로 물려받습니다.
그게 "이미 사진 보는 눈을 가진 감별사"를 데려오는 부분이에요.

```python
model.fc = nn.Linear(512, 2)   # 새로 만든 층 = 랜덤 초기화 = 반드시 학습 대상
```

`freeze_backbone=True`면 백본의 모든 파라미터에 `requires_grad = False`를 걸어
**업데이트를 막습니다.** 그러면 학습되는 건 새 `fc`의 1,026개(512×2 + 2)뿐입니다.
`fc` 교체를 동결 **뒤에** 하는 게 중요합니다. 그래야 새 층은 동결에서 빠집니다.

In [ ]:
def build_model(pretrained: bool, freeze_backbone: bool = False):
    weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = resnet18(weights=weights)          # weights=None 이면 랜덤 초기화

    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False            # 기존 층 전부 얼린다

    model.fc = nn.Linear(model.fc.in_features, 2)   # 이 층은 동결 이후에 만들어 항상 학습됨
    return model


demo = build_model(pretrained=True)
print("교체된 마지막 층:", demo.fc)
print(f"전체 파라미터: {sum(p.numel() for p in demo.parameters()):,}")

demo_frozen = build_model(pretrained=True, freeze_backbone=True)
print(f"백본 동결 시 학습 파라미터: {sum(p.numel() for p in demo_frozen.parameters() if p.requires_grad):,}")
del demo, demo_frozen

---
## 7. 학습·평가 루프

한 에폭(epoch) = 학습 데이터를 처음부터 끝까지 한 번 훑는 것. 배치마다 이 4단계를 반복합니다.

1. **순전파** — 사진을 넣어 예측을 얻는다
2. **손실 계산** — 정답과 얼마나 틀렸나 (`CrossEntropyLoss`)
3. **역전파** — 각 파라미터를 어느 방향으로 고쳐야 할지 계산 (`loss.backward()`)
4. **갱신** — 그 방향으로 조금 움직인다 (`optimizer.step()`)

몇 가지 선택에 대한 설명:

- **`AdamW`** — 학습률을 파라미터별로 알아서 조절해줘서 튜닝 부담이 적습니다.
- **`CosineAnnealingLR`** — 학습률을 후반으로 갈수록 부드럽게 줄입니다. 처음엔 성큼성큼,
  나중엔 조심조심 움직여 정답 근처에서 요동치지 않게 합니다.
- **`model.train()` / `model.eval()`** — BatchNorm과 Dropout의 동작을 바꿉니다.
  **평가 전에 `eval()`을 빠뜨리는 것이 가장 흔한 버그입니다.**
- **`torch.no_grad()`** — 평가 땐 기울기가 필요 없으니 계산을 꺼서 메모리와 시간을 아낍니다.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    # 정확도와 혼동행렬(confusion[정답][예측])을 함께 반환
    model.eval()
    correct = total = 0
    confusion = np.zeros((2, 2), dtype=int)
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total   += y.numel()
        for t, p in zip(y.cpu().numpy(), pred.cpu().numpy()):
            confusion[t, p] += 1
    return correct / max(total, 1), confusion


def run_experiment(name, desc, pretrained, freeze_backbone, lr, epochs=EPOCHS):
    set_seed(SEED)   # ★ 매 실험을 같은 출발점에서 — 차이가 '시작 가중치' 때문임을 보장

    model = build_model(pretrained, freeze_backbone).to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    n_train = sum(p.numel() for p in params)
    n_all   = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*66}\n[{name}] {desc}")
    print(f"  학습 파라미터 {n_train:,} / 전체 {n_all:,}   lr={lr}")

    history, started = [], time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        running, seen = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)   # 이전 기울기 지우기
            loss = criterion(model(x), y)           # 1~2 순전파 + 손실
            loss.backward()                         # 3 역전파
            optimizer.step()                        # 4 갱신
            running += loss.item() * y.size(0)
            seen += y.size(0)
        scheduler.step()

        acc, confusion = evaluate(model, test_loader, DEVICE)
        history.append({"epoch": epoch, "train_loss": running / seen, "test_acc": acc})
        print(f"  epoch {epoch}/{epochs}  train_loss={running/seen:.4f}  test_acc={acc*100:.2f}%")

    elapsed = time.time() - started
    final_acc, confusion = evaluate(model, test_loader, DEVICE)
    print(f"  → 최종 시험 정확도 {final_acc*100:.2f}%   ({elapsed:.1f}초)")
    print(f"    혼동행렬 [정답×예측]  Cat: {confusion[0].tolist()}   Dog: {confusion[1].tolist()}")

    return {"name": name, "desc": desc, "lr": lr, "trainable_params": n_train,
            "final_acc": final_acc, "seconds": elapsed,
            "confusion": confusion.tolist(), "history": history, "model": model}


results = {}
print("함수 준비 완료")

---
## 8. 실험 1 — 사전학습 파인튜닝 ⭐

ImageNet 가중치를 불러오고, `fc`만 2-클래스로 교체한 뒤 **전체를** 미세조정합니다.
학습률을 작게(`3e-4`) 주는 이유는, 크게 흔들면 애써 배워둔 특징이 망가지기 때문입니다.

In [ ]:
results["pretrained"] = run_experiment(
    "pretrained", "ImageNet 사전학습 + fc 교체 후 전체 파인튜닝",
    pretrained=True, freeze_backbone=False, lr=LR_PRETRAINED)

---
## 9. 실험 2 — 밑바닥부터 (비교군)

`weights=None`. 구조만 같고 숫자는 전부 랜덤입니다. **에폭 수, 데이터, 증강, 시드가 모두 동일**하고
학습률은 오히려 더 크게(`1e-3`) 줬습니다. 랜덤 초기화 쪽이 불리하지 않도록 배려한 설정입니다.

결과를 볼 때 **기준선(약 68%)** 을 꼭 같이 보세요. 68% 언저리면 모델이 "다 개"라고 찍고 있는 겁니다.

In [ ]:
results["scratch"] = run_experiment(
    "scratch", "랜덤 초기화, 밑바닥부터 학습",
    pretrained=False, freeze_backbone=False, lr=LR_SCRATCH)

---
## 10. 실험 3 (보너스) — 백본 동결, 마지막 층만 학습

전이학습의 효과가 정말 "이미 배운 특징" 덕분인지 확인하는 실험입니다.
백본을 통째로 얼려서 **1,026개 파라미터만** 학습시킵니다. 전체의 0.01%도 안 됩니다.

이것만으로도 성능이 잘 나온다면, ImageNet 특징이 이미 개/고양이를 구분할 정보를
충분히 담고 있다는 직접적인 증거가 됩니다.

In [ ]:
results["linear-probe"] = run_experiment(
    "linear-probe", "사전학습 백본 동결, 교체한 fc만 학습",
    pretrained=True, freeze_backbone=True, lr=LR_LINEAR)

---
## 11. 결과 비교

In [ ]:
order = ["pretrained", "scratch", "linear-probe"]
rows = [results[k] for k in order if k in results]

print(f"시험 {len(test_ds)}장 · {EPOCHS} epoch · 장치 {DEVICE}")
print(f"기준선(항상 Dog) = {BASELINE*100:.2f}%\n")
print(f"{'조건':<15}{'시험 정확도':>12}{'기준선 대비':>12}{'학습 파라미터':>16}{'시간(s)':>10}")
print("-" * 66)
for r in rows:
    print(f"{r['name']:<15}{r['final_acc']*100:>11.2f}%{(r['final_acc']-BASELINE)*100:>+11.2f}p"
          f"{r['trainable_params']:>16,}{r['seconds']:>10.1f}")

gap = results["pretrained"]["final_acc"] - results["scratch"]["final_acc"]
print(f"\n★ 사전학습 − 스크래치 = {gap*100:+.2f}%p")

In [ ]:
# 에폭별 시험 정확도 곡선
# (Colab에 한글 폰트가 없어 그래프 글자는 영어로 씁니다)
plt.figure(figsize=(8, 5))
for r in rows:
    xs = [h["epoch"] for h in r["history"]]
    ys = [h["test_acc"] * 100 for h in r["history"]]
    plt.plot(xs, ys, marker="o", label=f"{r['name']} (final {r['final_acc']*100:.1f}%)")

plt.axhline(BASELINE * 100, ls="--", c="gray", label=f"baseline: always Dog ({BASELINE*100:.1f}%)")
plt.xlabel("epoch"); plt.ylabel("test accuracy (%)")
plt.title("ResNet18 Cat vs Dog — pretrained vs scratch")
plt.xticks(range(1, EPOCHS + 1)); plt.ylim(40, 101)
plt.grid(alpha=0.3); plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(Path(OUT_DIR) / "comparison.png", dpi=140)
plt.show()

In [ ]:
# 혼동행렬 — 정확도 한 숫자만 보면 놓치는 것을 보여준다
fig, axes = plt.subplots(1, len(rows), figsize=(4.2 * len(rows), 3.8))
axes = np.atleast_1d(axes)
for ax, r in zip(axes, rows):
    cm = np.array(r["confusion"])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=13)
    ax.set_xticks([0, 1], CLASS_NAMES); ax.set_yticks([0, 1], CLASS_NAMES)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(f"{r['name']}  ({r['final_acc']*100:.1f}%)")
plt.tight_layout()
plt.show()

print("대각선이 정답입니다. scratch 쪽은 Cat 행(윗줄)이 무너져 있을 가능성이 큽니다")
print("→ 소수 클래스인 고양이를 개로 몰아 찍는 전형적인 미학습 상태")

---
## 12. 틀린 사진 들여다보기

99%를 맞혀도 남은 1%가 왜 틀렸는지 보는 게 다음 개선의 실마리입니다.
사전학습 모델이 틀린 사진을 직접 확인해봅시다.

In [ ]:
@torch.no_grad()
def collect_mistakes(model, loader, limit=8):
    model.eval()
    wrong = []
    for x, y in loader:
        logits = model(x.to(DEVICE))
        prob = logits.softmax(1).cpu()
        pred = prob.argmax(1)
        for i in (pred != y).nonzero(as_tuple=True)[0]:
            wrong.append((x[i], y[i].item(), pred[i].item(), prob[i, pred[i]].item()))
            if len(wrong) >= limit:
                return wrong
    return wrong


mistakes = collect_mistakes(results["pretrained"]["model"], test_loader)
print(f"사전학습 모델의 오답 {len(mistakes)}장 (최대 8장까지)")

if mistakes:
    n = len(mistakes)
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 3.0))
    for ax, (img, true, pred, conf) in zip(np.atleast_1d(axes), mistakes):
        ax.imshow(denormalize(img))
        ax.set_title(f"true {CLASS_NAMES[true]}\npred {CLASS_NAMES[pred]} ({conf*100:.0f}%)", fontsize=9)
        ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("오답이 없습니다")

In [ ]:
# 결과를 파일로 남겨두기 (model 객체는 JSON으로 저장할 수 없으니 제외)
payload = {
    "device": str(DEVICE), "epochs": EPOCHS,
    "train_size": len(train_ds), "test_size": len(test_ds),
    "baseline": float(BASELINE),
    "results": [{k: v for k, v in r.items() if k != "model"} for r in rows],
}
out = Path(OUT_DIR) / "results.json"
out.write_text(json.dumps(payload, indent=2, ensure_ascii=False))
print("저장:", out.resolve())

---
## 13. 정리 — 무엇을 배웠나

**참고 실행 결과** (Apple M-series MPS · 학습 2,000장 · 시험 1,000장 · 5 에폭):

| 조건 | 시험 정확도 | 학습 파라미터 | 시간 |
|---|---|---|---|
| pretrained | **99.30%** | 11,177,538 | 46.6초 |
| scratch | **70.80%** | 11,177,538 | 39.2초 |
| linear-probe | 97.90% | 1,026 | 14.5초 |

기준선(항상 Dog) 67.80% · **사전학습 − 스크래치 = +28.50%p**

> 시드를 고정했지만 GPU/MPS 연산과 DataLoader 워커의 순서 때문에 **재실행 시 소수점 단위로
> 달라질 수 있습니다.** 실제로 같은 코드를 노트북에서 다시 돌렸을 때 pretrained 99.30% /
> scratch 71.20% / linear-probe 97.90% 가 나왔습니다. 결론(28%p 안팎의 격차)은 그대로입니다.

### 세 가지 교훈

**① 데이터가 적을 때 전이학습은 선택이 아니라 필수입니다.**
스크래치의 70.80%는 기준선보다 3%p 높을 뿐입니다. 혼동행렬을 보면 고양이 322장 중 145장만
맞혔어요. 사실상 "대체로 개"라고 찍는 상태입니다. 1,100만 개 파라미터를 2,000장으로
처음부터 가르치는 건 애초에 무리입니다.

**② 성능의 대부분은 백본이 이미 갖고 있었습니다.**
파라미터의 0.01%(1,026개)만 학습한 linear-probe가 97.90%입니다.
ImageNet에서 배운 "털·귀·눈 모양을 읽는 능력"이 그대로 개/고양이 구분에 쓰인 겁니다.
GPU 시간이 부족하다면 이 방법만으로도 충분할 때가 많습니다.

**③ 정확도 한 숫자만 믿지 마세요.**
클래스가 2:1로 치우친 데이터에서는 아무 판단 없이 찍어도 68%가 나옵니다.
**기준선과 혼동행렬**을 함께 봐야 모델이 진짜 학습했는지 알 수 있습니다.

### 더 해볼 것

- `TRAIN_SIZE = 0`, `EPOCHS = 30` — 데이터와 시간을 늘리면 스크래치가 얼마나 따라올까요?
  (따라오긴 하지만 끝내 못 이깁니다)
- `EPOCHS = 15`로 늘려 사전학습 모델의 **과적합** 관찰하기 — train_loss는 계속 떨어지는데
  test_acc는 멈추거나 떨어지는 지점이 보일 겁니다
- `resnet50`, `efficientnet_b0` 등 다른 백본과 비교
- `target_types="category"`로 바꿔 **37품종 분류**에 도전 (훨씬 어렵습니다)